In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../part3/data/sales.csv", parse_dates=["date"])
df["price"] = df["price"].fillna(df.groupby("product")["price"].transform("median"))

limit = df.groupby("product")["qty"].transform(lambda s: s.quantile(0.99))
df = df[df["qty"] <= limit].copy()

median_price = df.groupby("product")["price"].transform("median")
df = df[df["price"].between(0.5 * median_price, 2 * median_price)].copy()

df["revenue"] = df["qty"] * df["price"]
print(df.shape)

(58148, 11)


In [2]:
weekly = (df.set_index("date")
            .groupby("category")["revenue"]
            .resample("W")
            .sum()
            .unstack(level=0))

weekly = weekly.iloc[1:-1]
print(weekly.shape)
weekly.head()

(156, 3)


category,Напитки,Одежда,Электроника
date,,,
2022-01-09,936827.400,4114917.065,3317401.34
2022-01-16,941823.605,4319651.480,3366624.06
2022-01-23,925723.680,4168110.980,3344000.40
2022-01-30,923961.060,4137697.360,3181329.17
2022-02-06,952474.970,3964332.710,3519001.25


In [3]:
TEST_WEEKS = 8
train = weekly.iloc[:-TEST_WEEKS]
test = weekly.iloc[-TEST_WEEKS:]
print("обучение:", len(train), "недель, тест:", len(test), "недель")

обучение: 148 недель, тест: 8 недель


In [4]:
def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

forecasts = {} 
results = []

for category in weekly.columns:
    actual = test[category].values
    all_sales = weekly[category]
    naive = all_sales.shift(52).iloc[-TEST_WEEKS:].values
    ma = np.full(TEST_WEEKS, train[category].iloc[-4:].mean())

    forecasts[(category, "сезонный наивный")] = naive
    forecasts[(category, "скользящее среднее")] = ma

    results.append({"категория": category,
                    "MAE сезонный наивный": mae(actual, naive),
                    "MAE скользящее среднее": mae(actual, ma)})

results = pd.DataFrame(results).round(0)
results

,категория,MAE сезонный наивный,MAE скользящее среднее
0,Напитки,62243.0,30078.0
1,Одежда,252263.0,613522.0
2,Электроника,409031.0,232506.0


In [ ]:
fig, axes = plt.subplots(len(weekly.columns), 1, figsize=(10, 4 * len(weekly.columns)))

for ax, category in zip(axes, weekly.columns):
    ax.plot(train.index[-20:], train[category].iloc[-20:], color="gray", label="обучение (последние 20 недель)")
    ax.plot(test.index, test[category], "o-", color="black", label="факт")
    ax.plot(test.index, forecasts[(category, "сезонный наивный")], "s--", label="сезонный наивный")
    ax.plot(test.index, forecasts[(category, "скользящее среднее")], "^--", label="скользящее среднее")
    ax.set_title(category)
    ax.set_ylabel("выручка за неделю, руб.")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()